# EfficientNet-B4 Fine-Tuning: RDD2022 Road Defect Classifier

This notebook trains an EfficientNet-B4 model to classify road images into five categories:

| Class | Label |
|-------|-------|
| 0 | Background (no defect) |
| 1 | D00 - Pothole |
| 2 | D10 - Longitudinal crack |
| 3 | D20 - Transverse crack |
| 4 | D40 - Alligator crack |

**Why EfficientNet-B4?**
EfficientNet scales a CNN's depth, width, and input resolution together using a fixed
compound coefficient. B4's 380x380 native input resolution captures fine-grained crack
patterns that smaller variants miss, without requiring the compute of B5 or larger.

**Why fine-tune instead of train from scratch?**
ImageNet pre-training gives the model edge detectors and texture features for free.
Fine-tuning on RDD2022 reuses those features and adapts only the final layers to the
defect domain. This converges faster and needs far fewer labeled examples.

---
Make sure you have downloaded the dataset first:
```bash
python data/download_rdd2022.py
```

## 0. Imports and setup

In [1]:
import sys
sys.path.insert(0, '..')  # let the notebook find the project root

import matplotlib.pyplot as plt
import torch
import timm
from pathlib import Path
from torch import nn, optim
from torch.utils.data import ConcatDataset, DataLoader, random_split
from torchvision import transforms
from tqdm.notebook import tqdm

from config import (
    set_seeds, SEED,
    EFFICIENTNET_MODEL,
    EFFICIENTNET_LR,
    EFFICIENTNET_WEIGHT_DECAY,
    EFFICIENTNET_BATCH_SIZE,
    EFFICIENTNET_EPOCHS,
    EFFICIENTNET_IMG_SIZE,
    EFFICIENTNET_NUM_WORKERS,
    RDD2022_NUM_CLASSES,
    RDD2022_DIR,
    RDD2022_CHECKPOINT,
    CHECKPOINT_DIR,
)
from src.datasets.rdd2022 import RDD2022Dataset

set_seeds(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## 1. Load the dataset

We combine all four country splits (Japan, India, Czech, Norway) into one pool,
then hold out 15% as a validation set. Training images get random flips and
color jitter; validation images get only the resize and normalize.

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((EFFICIENTNET_IMG_SIZE, EFFICIENTNET_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((EFFICIENTNET_IMG_SIZE, EFFICIENTNET_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

countries = ['Japan', 'India', 'Czech', 'Norway']
all_datasets = []
missing_countries = []
for country in countries:
    country_dir = RDD2022_DIR / country
    if country_dir.is_dir():
        dataset = RDD2022Dataset(str(country_dir), split='train', transform=train_transform)
        if len(dataset) == 0:
            raise RuntimeError(
                f'RDD2022 {country} directory exists but contains no labeled training samples: {country_dir}'
            )
        all_datasets.append(dataset)
    else:
        missing_countries.append(country)

if missing_countries:
    missing = ' '.join(missing_countries)
    raise FileNotFoundError(
        f'Missing RDD2022 country directories: {missing_countries}. '
        f'Run "python data/download_rdd2022.py --countries {missing}" '
        f'from the project root ({RDD2022_DIR.parent.parent.parent}).'
    )

full_dataset = ConcatDataset(all_datasets)

val_size = int(0.15 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=EFFICIENTNET_BATCH_SIZE,
    shuffle=True,
    num_workers=EFFICIENTNET_NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=EFFICIENTNET_BATCH_SIZE,
    shuffle=False,
    num_workers=EFFICIENTNET_NUM_WORKERS,
    pin_memory=True,
)

print(f'Training samples:   {train_size:,}')
print(f'Validation samples: {val_size:,}')

FileNotFoundError: Missing RDD2022 country directories: ['Japan', 'India', 'Czech', 'Norway']. Run "python data/download_rdd2022.py --countries Japan India Czech Norway" from the project root (c:\Users\joshu\Documents\VS Code\defect_detector\defect-detector\notebooks\..).

## 1.1 Class distribution

Before building the model it is worth checking how the labels break down across all four
country splits. Background images (no defect annotation) tend to dominate datasets like
this, which directly motivates using per-class metrics rather than accuracy alone.

In [ ]:
from collections import Counter
from src.datasets.rdd2022 import parse_annotation

CLASS_NAMES = {
    0: 'Background',
    1: 'Pothole',
    2: 'Long. crack',
    3: 'Trans. crack',
    4: 'Alligator crack',
}

label_counts: Counter = Counter()
for country in countries:
    ann_dir = RDD2022_DIR / country / 'train' / 'annotations' / 'xmls'
    if not ann_dir.is_dir():
        continue
    for xml_path in sorted(ann_dir.glob('*.xml')):
        anns = parse_annotation(xml_path)
        if not anns:
            label_counts[0] += 1
        else:
            raw_labels = [a['label'] for a in anns]
            most_frequent = max(set(raw_labels), key=raw_labels.count)
            label_counts[most_frequent] += 1

classes = sorted(CLASS_NAMES.keys())
counts = [label_counts[c] for c in classes]
names = [CLASS_NAMES[c] for c in classes]
total = sum(counts)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, counts, color='steelblue', edgecolor='white')
ax.set_ylabel('Images')
ax.set_title('RDD2022 training set - image-level class distribution')
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 150,
        f'{count:,}\n({count / total:.1%})',
        ha='center', va='bottom', fontsize=9,
    )
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.savefig('class_distribution_rdd2022.png', dpi=120)
plt.show()
print(f'Total training images: {total:,}')

## 2. Build the model

We load EfficientNet-B4 pre-trained on ImageNet and replace its classifier
head with a new linear layer sized to our 5-class problem.
AdamW is used with a step decay scheduler that drops the learning rate
by 10x halfway through training.

In [ ]:
model = timm.create_model(
    EFFICIENTNET_MODEL,
    pretrained=True,
    num_classes=RDD2022_NUM_CLASSES,
)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=EFFICIENTNET_LR,
    weight_decay=EFFICIENTNET_WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=EFFICIENTNET_EPOCHS // 2,
    gamma=0.1,
)

## 3. Training loop

Each epoch runs one pass over the training set and one pass over the
validation set. The best checkpoint (by validation accuracy) is saved
to disk so we can reload it for evaluation.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """One pass over the training set. Returns (avg loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate on a data loader. Returns (avg loss, accuracy)."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val  ', leave=False):
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)

history: dict[str, list[float]] = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
}
best_val_acc = 0.0

for epoch in range(1, EFFICIENTNET_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch {epoch:02d}/{EFFICIENTNET_EPOCHS}  '
        f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
        f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), RDD2022_CHECKPOINT)
        print(f'  Saved best checkpoint (val_acc={best_val_acc:.3f})')

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.3f}')

## 4. Training curves

In [ ]:
epochs = range(1, EFFICIENTNET_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Cross-Entropy Loss')
ax1.legend()

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves_rdd2022.png', dpi=120)
plt.show()
print('Saved training_curves_rdd2022.png')

## 5. Per-class metrics on the validation set

Overall accuracy can mask a model that learned to predict the majority class.
Per-class precision, recall, and F1 give a clearer picture of where the
model is strong and where it struggles.

In [ ]:
from sklearn.metrics import classification_report

# Reload the best checkpoint before evaluating.
model.load_state_dict(torch.load(RDD2022_CHECKPOINT, map_location=DEVICE))
model.eval()

all_preds: list[int] = []
all_labels: list[int] = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating'):
        logits = model(images.to(DEVICE))
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

class_names = ['Background', 'Pothole', 'Longitudinal crack', 'Transverse crack', 'Alligator crack']
print(classification_report(all_labels, all_preds, target_names=class_names))